# Neural-network basics

Exercise notebook for the lesson **[Neural-network basics](https://ramkumarmd.com/medical-ai/deep-learning-imaging/neural-network-basics)** in the *Deep learning & imaging* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** A neural network is a chain of simple layers: each multiplies its inputs by learned weights, adds them up and passes the result through a non-linear function. Stacking layers lets the network learn curved, complex relationships that a linear model such as logistic regression cannot. It learns by repeatedly comparing its predictions with the correct answers and nudging every weight to reduce the error, a process called gradient descent with backpropagation. Frameworks such as PyTorch do the calculus automatically.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Data a straight line cannot separate

Two interleaved, curved classes: logistic regression as the baseline.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=2000, noise=0.25, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
lr = LogisticRegression().fit(X_train, y_train)
print(f"Logistic regression accuracy: {lr.score(X_test, y_test):.3f}")

# Logistic regression accuracy: 0.852

### A neural network from scratch

Continuing from the previous block: a two-layer network in NumPy.

In [ ]:
rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, 1, (2, 16)), np.zeros(16)          # input -> 16 hidden units
W2, b2 = rng.normal(0, 1, (16, 1)), np.zeros(1)           # hidden -> output
sigmoid = lambda z: 1 / (1 + np.exp(-z))

for epoch in range(2001):
    h = np.tanh(X_train @ W1 + b1)                         # forward pass
    p = sigmoid(h @ W2 + b2).ravel()
    grad_out = (p - y_train)[:, None] / len(y_train)       # backward pass (cross-entropy loss)
    grad_W2, grad_b2 = h.T @ grad_out, grad_out.sum(0)
    grad_h = grad_out @ W2.T * (1 - h ** 2)
    grad_W1, grad_b1 = X_train.T @ grad_h, grad_h.sum(0)
    for param, grad in [(W1, grad_W1), (b1, grad_b1), (W2, grad_W2), (b2, grad_b2)]:
        param -= 1.0 * grad                                # gradient descent step
    if epoch % 500 == 0:
        loss = -np.mean(y_train * np.log(p + 1e-9) + (1 - y_train) * np.log(1 - p + 1e-9))
        print(f"epoch {epoch:>4}  training loss {loss:.3f}")

p_test = sigmoid(np.tanh(X_test @ W1 + b1) @ W2 + b2).ravel()
print(f"Neural network accuracy: {np.mean((p_test > 0.5) == y_test):.3f}")

# epoch    0  training loss 0.915
# epoch  500  training loss 0.125
# epoch 1000  training loss 0.122
# epoch 1500  training loss 0.121
# epoch 2000  training loss 0.120
# Neural network accuracy: 0.938

### The same network in PyTorch

Continuing from the first block: the network in PyTorch (pip install torch).

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1))
optimiser = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
Xt, yt = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)

for epoch in range(500):
    optimiser.zero_grad()
    loss = loss_fn(model(Xt).squeeze(1), yt)
    loss.backward()                                          # PyTorch computes the gradients
    optimiser.step()

with torch.no_grad():
    logits = model(torch.tensor(X_test, dtype=torch.float32)).squeeze(1)
print(f"PyTorch network accuracy: {((logits > 0).numpy() == y_test).mean():.3f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

# PyTorch network accuracy: 0.945
# Parameters: 65

## Exercise

**Goal:** Build, train and probe a small neural network, and see when it helps over logistic regression.

**Data:** Synthetic data generated in the notebook (scikit-learn make_moons) (Generated in the notebook)

### Step 1

Reproduce the NumPy and PyTorch networks and plot their decision boundaries.

In [ ]:
# Your code here


### Step 2

Change the hidden layer to 2, 4 and 64 units and compare test accuracy and boundaries.

In [ ]:
# Your code here


### Step 3

Add more noise to the data and show how a large network can overfit; add early stopping on a validation set.

In [ ]:
# Your code here


### Step 4

Train the PyTorch network on the synthetic diabetes-risk data from the logistic-regression lesson and compare AUROC and calibration with logistic regression.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/neural-network-basics](https://ramkumarmd.com/medical-ai/deep-learning-imaging/neural-network-basics). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.